# 04 — Land Use Mix

Computes Shannon entropy of PLUTO land use distribution per grid cell to measure how mixed vs. homogeneous each cell is.

**Data source:** PLUTO CSV (NYC only — `needs_pluto`).

**Note:** Raw area ratios (`comarea`, `resarea`, etc.) are intentionally excluded to prevent Y variable leakage. HHI dropped (r=-0.92 with entropy).

**Output columns:** `cell_id`, `landuse_entropy`

**Output file:** `csv/04_land_use_mix.csv`

In [ ]:
# ── Papermill parameters ──────────────────────────────
GRID_CONFIG = "grid.json"

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math

with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CSV_DIR = config.get("csv_dir", "csv")
os.makedirs(CSV_DIR, exist_ok=True)

if not config["feature_flags"].get("needs_pluto", False):
    print("PLUTO not available — skipping notebook 04.")
    df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
    df_empty = pd.DataFrame({"cell_id": df_grid["cell_id"]})
    df_empty["landuse_entropy"] = np.nan
    df_empty.to_csv(f"{CSV_DIR}/04_land_use_mix.csv", index=False)
    raise SystemExit("Skipped — needs_pluto=false")

PLUTO_PATH = config["pluto_path"]
BOROUGH_CODES = config["borough_codes"]
BOROUGH_FILTER = config["borough_filter"]
CELL_SIZE_M = config["grid_cell_size_m"]
boro_code_filter = [str(BOROUGH_CODES[b]) for b in BOROUGH_FILTER]
print(f"Loading PLUTO from {PLUTO_PATH}")

In [ ]:
# ── Load PLUTO + assign to grid cells ─────────────────
COLS = ["borocode", "landuse", "lotarea", "latitude", "longitude"]

df_pluto = pd.read_csv(PLUTO_PATH, usecols=COLS, dtype={"landuse": str})
df_pluto = df_pluto[df_pluto["borocode"].astype(str).isin(boro_code_filter)].copy()
df_pluto["lotarea"] = pd.to_numeric(df_pluto["lotarea"], errors="coerce").fillna(0)
df_pluto["latitude"] = pd.to_numeric(df_pluto["latitude"], errors="coerce")
df_pluto["longitude"] = pd.to_numeric(df_pluto["longitude"], errors="coerce")
df_pluto = df_pluto.dropna(subset=["latitude", "longitude"]).copy()

# Grid parameters — must match notebook 01
REF_LAT = df_pluto["latitude"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
BUFFER = LAT_STEP
LAT_MIN = df_pluto["latitude"].min() - BUFFER
LON_MIN = df_pluto["longitude"].min() - BUFFER

df_pluto["grid_row"] = ((df_pluto["latitude"] - LAT_MIN) / LAT_STEP).astype(int)
df_pluto["grid_col"] = ((df_pluto["longitude"] - LON_MIN) / LON_STEP).astype(int)
df_pluto["cell_id"] = "r" + df_pluto["grid_row"].astype(str).str.zfill(4) + "_c" + df_pluto["grid_col"].astype(str).str.zfill(4)

# Only keep lots in valid grid cells
df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
valid_cells = set(df_grid["cell_id"])
df_pluto = df_pluto[df_pluto["cell_id"].isin(valid_cells)].copy()
print(f"PLUTO lots in valid grid cells: {len(df_pluto):,}")
print(f"Unique landuse codes: {df_pluto['landuse'].nunique()}")

In [ ]:
# ── Compute Shannon entropy per grid cell ─────────────

def shannon_entropy(proportions):
    """Shannon entropy from a list of proportions (0-1)."""
    proportions = proportions[proportions > 0]
    if len(proportions) == 0:
        return 0.0
    return -np.sum(proportions * np.log2(proportions))


records = []

for cell_id, group in df_pluto.groupby("cell_id"):
    total_area = group["lotarea"].sum()
    if total_area == 0:
        records.append({"cell_id": cell_id, "landuse_entropy": 0.0})
        continue
    
    lu_area = group.groupby("landuse")["lotarea"].sum()
    proportions = (lu_area / total_area).values
    
    records.append({
        "cell_id": cell_id,
        "landuse_entropy": round(shannon_entropy(proportions), 4),
    })

df_mix = pd.DataFrame(records)

# Ensure all grid cells are present
df_result = df_grid[["cell_id"]].merge(df_mix, on="cell_id", how="left")
print(f"Computed entropy for {len(df_result)} cells")
print(f"Entropy: mean={df_result['landuse_entropy'].mean():.3f}, "
      f"min={df_result['landuse_entropy'].min():.3f}, "
      f"max={df_result['landuse_entropy'].max():.3f}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/04_land_use_mix.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)